# Chapter 07 — LLM Training Configuration

You have the architecture — now you need to train it well. Optimization is the art and science of getting neural networks to converge quickly and reliably. Poor optimization choices can waste weeks of GPU time or prevent the model from learning at all. This chapter covers the essential techniques that every LLM practitioner must master.

### Glossary / Glossário

• AdamW — Adam optimizer with decoupled weight decay, the standard for LLM training. / Otimizador Adam com weight decay desacoplado, padrão para treinamento de LLMs.
• SGD — Stochastic Gradient Descent, basic optimizer updating parameters with gradient × learning rate. / Gradiente descendente estocástico, otimizador básico que atualiza parâmetros com gradiente × learning rate.
• warmup — initial training phase where learning rate gradually increases from near-zero. / Fase inicial do treinamento onde learning rate aumenta gradualmente a partir de quase zero.
• cosine decay — learning rate schedule following a cosine curve from peak to minimum. / Schedule de learning rate seguindo curva cosseno do pico ao mínimo.
• gradient clipping — capping the global gradient norm to prevent exploding gradients. / Limitação da norma global do gradiente para prevenir gradientes explosivos.
• weight decay — regularization that penalizes large weights by shrinking them each step. / Regularização que penaliza pesos grandes encolhendo-os a cada passo.

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

model = nn.Linear(256, 256)

optimizer = AdamW(
    model.parameters(),
    lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-8,
)

warmup_steps = 2000
total_steps = 100000
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
cosine = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=3e-5)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_steps])

print(f"{'Step':>7s} | {'Learning Rate':>14s} | {'Phase':<12s}")
print("-" * 40)
checkpoints = [0, 500, 1000, 1999, 2000, 10000, 50000, 99999]
for step in range(total_steps):
    optimizer.step()
    scheduler.step()
    if step in checkpoints:
        lr = optimizer.param_groups[0]['lr']
        phase = "warmup" if step < warmup_steps else "cosine decay"
        print(f"{step:>7d} | {lr:>14.6f} | {phase:<12s}")

In [ ]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# AdamW optimizer with LLM-standard hyperparameters
optimizer = AdamW(
    model.parameters(),
    lr=3e-4,            # Peak learning rate
    betas=(0.9, 0.95),  # Momentum parameters
    weight_decay=0.1,   # Regularization
    eps=1e-8,
)

# Warmup + Cosine Decay schedule
warmup = LinearLR(optimizer, start_factor=0.01, total_iters=2000)
cosine = CosineAnnealingLR(optimizer, T_max=100000, eta_min=3e-5)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[2000])

# Training loop with gradient clipping
for batch in dataloader:
    logits = model(batch["input_ids"])
    loss = F.cross_entropy(logits.view(-1, vocab_size), batch["labels"].view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

---

**Course:** Aprenda LLM | **Chapter 07**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.